In [2]:
from webscrape_utils import read_credentials, write_string_to_s3, setup_driver, get_aws_session, upload_to_s3, write_to_local_file
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from botocore.exceptions import BotoCoreError, ClientError
from datetime import datetime
import json
import time
import boto3
import os

country = 'ar'

def setup_driver():
    options = ChromeOptions()
    #options.add_argument("--headless")
    options.add_argument("--window-size=2048,2048")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-software-rasterizer")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    options.binary_location = "/usr/bin/chromium-browser"

    capabilities = DesiredCapabilities.CHROME.copy()
    capabilities['goog:chromeOptions'] = {
        'prefs': {
            'profile.default_content_setting_values.geolocation': 1
        }
    }

    service = ChromeService(executable_path="/usr/lib/chromium-browser/chromedriver")
    driver = webdriver.Chrome(service=service, options=options)

    driver.execute_cdp_cmd("Emulation.setGeolocationOverride", {
        "latitude": -34.56060895367534,
        "longitude": -58.45812919702398,
        "accuracy": 100
    })

    return driver

driver = setup_driver()

wait = WebDriverWait(driver, 300)  # Adjust the timeout as needed

url = 'https://www.mcdonalds.com.ar/restaurantes/ciudad-autonoma-de-buenos-aires/cabildo-y-olazabal-bel/pedidos/pedi-y-retira/hamburguesas'
driver.get(url)

In [6]:
big_mac_price = price_element = wait.until(EC.presence_of_element_located(
(By.XPATH, "//p[text()='Big Mac']/following-sibling::div")))

big_mac_price = big_mac_price.text.replace("$", "").replace(",", ".").replace(".", "")    

big_mac_price

' 730000'

In [1]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime, timedelta
from time import sleep
import time
import json
import os

country = 'ar'

# Get the user's home directory
home_directory = os.path.expanduser('~')

# To address an issue caused by the Snap installation of Firefox
#tmp_dir = os.path.join(home_directory, 'tmp')
#os.environ['TMPDIR'] = tmp_dir
#os.system(f"rm -rf {tmp_dir}/*")

#print(f"Does this command look right? rm -rf {tmp_dir}/*")

options = FirefoxOptions()
# options.binary = "/opt/homebrew/bin/firefox"
#options.add_argument("--headless")  # Run Firefox in headless mode

options.add_argument("--width=2048")
options.add_argument("--height=2048")

# Set preferences for geolocation
options.set_preference("geo.enabled", True)
options.set_preference("geo.prompt.testing", True)
options.set_preference("geo.prompt.testing.allow", False)  # Set to True to allow geolocation

# service = FirefoxService(executable_path=os.path.join(home_directory, "airflow/geckodriver"))

driver = webdriver.Firefox(options=options)

size = driver.get_window_size()
#print("Window size: width = {}px, height = {}px".format(size["width"], size["height"]))

wait = WebDriverWait(driver, 300)  # Adjust the timeout as needed

url = 'https://www.mcdonalds.com.ar/pedidos/seleccionar-restaurante'

driver.get(url)

NoSuchDriverException: Message: Unable to obtain driver for firefox; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


In [72]:
addr_input = wait.until(EC.element_to_be_clickable((By.XPATH, "//input[@placeholder='Escribe tu ciudad o tu dirección']")))
addr_input.send_keys("Avenida Cabildo 2254, Buenos Aires")
addr_input.send_keys(Keys.RETURN)

In [73]:
restaurant_selection_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".mcd-restaurant-d-actions .button.is-primary")))
restaurant_selection_button.click()

In [74]:
hamburguesas = wait.until(EC.element_to_be_clickable((By.XPATH, f"//div[contains(text(), 'Hamburguesas')]")))

hamburguesas.click()

In [75]:
hamburguesas = wait.until(EC.element_to_be_clickable((By.XPATH, f"//div[contains(text(), 'Big Mac')]")))

hamburguesas.click()

In [76]:
big_mac_div = wait.until(EC.visibility_of_element_located((By.XPATH, '//h4[contains(text(), "Big Mac")]')))
big_mac_price = big_mac_div.find_element(By.XPATH, "following-sibling::h5")
big_mac_price = big_mac_price.text.replace("$", "").replace(",", ".").replace(".", "")
big_mac_price

' 370000'

In [77]:
driver.quit()